In [ ]:
# Step 0: Installs
!pip -q install "openai>=1.40.0" \
  "chromadb<=0.5.23" \
  "tokenizers<=0.20.3" \
  "httpx<0.28.0" \
  tiktoken pypdf python-docx beautifulsoup4 html2text requests \
  pandas openpyxl

In [ ]:
# Step 1: Imports & Config
import os, io, re, hashlib, zipfile, tempfile, urllib.parse, requests, logging, json, time, textwrap, sys
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd

# Chroma telemetry off before import
os.environ["CHROMA_TELEMETRY_IMPLEMENTATION"] = "none"
os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb.telemetry").setLevel(logging.CRITICAL)
logging.getLogger("chromadb").setLevel(logging.WARNING)

import chromadb
import tiktoken
from openai import OpenAI
from pypdf import PdfReader
from docx import Document
from bs4 import BeautifulSoup
import html2text

def _dbg(msg: str):
    print(f"[DEBUG] {msg}")

# Caches
_HTTP_CACHE: dict[str, str] = {}                  # url -> html ("" sentinel for non-HTML)
_SERPER_CACHE: dict[tuple, list[dict]] = {}       # (q, num) -> hits
_EMBED_CACHE_Q: dict[str, list[float]] = {}       # query text -> vec
_EMBED_CACHE_DOC: dict[str, list[float]] = {}     # doc chunk text -> vec
# (question, W_MAX, P_MAX, PER_DOMAIN, tuple(domains), tuple(names)) -> {"domain":[...], "people":[...]}
_WEB_SNIP_CACHE: dict[tuple, dict[str, list[dict]] ] = {}

# Secrets helpers
def _maybe_get_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

def _ensure_secret(env_name: str, colab_key_names: list[str], prompt_text: str = "") -> str:
    val = (os.environ.get(env_name) or "").strip()
    if not val:
        for k in colab_key_names:
            v = _maybe_get_colab_secret(k)
            if v:
                val = v.strip()
                break
    if not val and prompt_text:
        try:
            import getpass
            entered = getpass.getpass(prompt_text)
            val = (entered or "").strip()
        except Exception:
            pass
    if val:
        os.environ[env_name] = val
    return val

def _serper_preflight(key: str) -> bool:
    if not key:
        print("❌ SERPER_API_KEY missing.")
        return False
    try:
        r = requests.post(
            "https://google.serper.dev/search",
            headers={"X-API-KEY": key, "Content-Type": "application/json"},
            json={"q": "site:example.com example", "num": 1, "gl": "us", "hl": "en"},
            timeout=15,
        )
        if r.status_code == 200:
            print("✅ Serper preflight OK.")
            return True
        if r.status_code in (401, 403):
            print(f"❌ Serper auth error {r.status_code}. Check the key/plan.")
            return False
        print(f"⚠️ Serper non-200 {r.status_code}: {(r.text or '')[:160]}")
        return False
    except Exception as e:
        print("⚠️ Serper preflight error:", e)
        return False

# Keys
OPENAI_API_KEY = _ensure_secret("OPENAI_API_KEY", ["open_ai_key","OPENAI_API_KEY"], "Enter your OPENAI_API_KEY: ")
if not OPENAI_API_KEY or OPENAI_API_KEY == "YOUR_OPENAI_API_KEY":
    raise RuntimeError("Set OPENAI_API_KEY (Colab: Runtime ▸ Secrets ▸ add key 'open_ai_key').")
SERPER_API_KEY = _ensure_secret("SERPER_API_KEY", ["serper_api_key","SERPER_API_KEY"])
print("OPENAI_API_KEY present?:", bool(OPENAI_API_KEY))
print("SERPER_API_KEY present?:", bool(SERPER_API_KEY))
if SERPER_API_KEY:
    print("SERPER_API_KEY suffix:", "****" + SERPER_API_KEY[-4:])
    _serper_preflight(SERPER_API_KEY)

# Dropbox links
KB_SHARED_URL    = "https://www.dropbox.com/scl/fo/nybhgbl5b2831vguyhqvp/ANMo-KwWTbo3tW3DZ8oQpV4?rlkey=5gtf1syx3kgrqh6oefnnz9394&e=3&st=q7wczfem&dl=0"
STYLE_SHARED_URL = "https://www.dropbox.com/scl/fo/ync0oj4ycqluc9lhl2p01/AC6Md4a4rif-Q1sTS0P5XG0?rlkey=yjj7o0lerh4etmrff8ymzlf8w&e=2&st=4dsln4gd&dl=0"
EXTERNAL_SHARED_URL = "https://www.dropbox.com/scl/fo/oows8fqeh7pxfpwlk2g3g/APvqEeOtb2x7zfNK8WB03eM?rlkey=lddamawbvgde5fwwnbd9ykzo0&st=wbofa3f3&dl=0"

EXTERNAL_EXCEL_PATH = "/content/Chatbot Trusted Sources.xlsx"

# Models & knobs
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL  = "gpt-4o-mini"

KB_CHUNK_SIZE      = 1200
KB_CHUNK_OVERLAP   = 200

# Keep K# unlimited as requested earlier
KB_TOP_K: Optional[int] = None       # None => return all available K# chunks

STYLE_TOP_K        = 3
MIN_CONTEXT_TOKENS = 400

# Web limits (user knobs)
WEB_DOMAIN_MAX = 5   # up to 5 domain snippets (W#)
WEB_PEOPLE_MAX = 3   # up to 3 people-authored snippets (P#)
WEB_PER_DOMAIN = 1   # diversity: max N per domain
WEB_TOTAL_MAX  = 5   # HARD CAP across W# + P# per response

# Collections
KB_COLLECTION_NAME       = "dropbox_kb"
STYLE_COLLECTION_NAME    = "dropbox_style"
EXTERNAL_COLLECTION_NAME = "dropbox_external"

TRUSTED_SOURCES = {"names": [], "domains": []}

OPENAI_API_KEY present?: True
SERPER_API_KEY present?: True
SERPER_API_KEY suffix: ****ef8d
✅ Serper preflight OK.


In [ ]:
# Step 2: Clients (OpenAI + Chroma)

from openai import OpenAI
import chromadb

# OpenAI client (uses OPENAI_API_KEY from Step 1)
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Chroma client: prefer in-memory; fallback to on-disk if needed
try:
    chroma_client = chromadb.EphemeralClient()
    chroma_mode = "EphemeralClient (in-memory)"
except Exception:
    chroma_client = chromadb.PersistentClient(path="/content/chroma")
    chroma_mode = "PersistentClient (/content/chroma)"

# Create/get collections
kb_collection = chroma_client.get_or_create_collection(
    name=KB_COLLECTION_NAME, metadata={"source": "dropbox_shared"}
)
style_collection = chroma_client.get_or_create_collection(
    name=STYLE_COLLECTION_NAME, metadata={"source": "dropbox_shared"}
)
external_collection = chroma_client.get_or_create_collection(
    name=EXTERNAL_COLLECTION_NAME, metadata={"source": "dropbox_shared_external"}
)

# Quick visibility
print("OpenAI ready:", bool(os.environ.get("OPENAI_API_KEY")))
print("Chroma mode:", chroma_mode)
print("KB ok:", kb_collection is not None, "size:", kb_collection.count())
print("Style ok:", style_collection is not None, "size:", style_collection.count())
print("External ok:", external_collection is not None, "size:", external_collection.count())

# Optional: quick OpenAI preflight (no content cost beyond a tiny request)
try:
    _ = client.models.list()
    print("✅ OpenAI preflight OK.")
except Exception as e:
    print("❌ OpenAI preflight failed:", e)

OpenAI ready: True
Chroma mode: EphemeralClient (in-memory)
KB ok: True size: 36
Style ok: True size: 7
External ok: True size: 0
✅ OpenAI preflight OK.


In [ ]:
# Step 3: Helpers (tokenizer, chunking, cleaners) — optimized + cached

import hashlib
import re
import tiktoken
from functools import lru_cache
from typing import List

# stable id for chunks
def sha1(text: str) -> str:
    return hashlib.sha1((text or "").encode("utf-8")).hexdigest()

# cache tokenizer instance
@lru_cache(maxsize=1)
def get_encoder():
    return tiktoken.get_encoding("cl100k_base")

# precompiled whitespace regex
_WS_RE = re.compile(r"\s+")

def clean_text(s: str) -> str:
    # simple, fast whitespace collapse
    return _WS_RE.sub(" ", (s or "")).strip()

@lru_cache(maxsize=4096)
def count_tokens(text: str) -> int:
    # disallow no specials ⇒ slightly faster encode
    return len(get_encoder().encode(text or "", disallowed_special=()))

@lru_cache(maxsize=1024)
def _chunk_text_cached(text: str, chunk_size: int, overlap: int) -> tuple:
    """
    Internal cached chunker. Returns a tuple for cacheability.
    """
    text = text or ""
    enc = get_encoder()
    toks = enc.encode(text, disallowed_special=())
    n = len(toks)
    if n == 0:
        return ()
    # fast-path: already small
    if n <= chunk_size:
        return (text.strip(),)

    # ensure forward progress even if overlap is large
    step = max(1, chunk_size - max(0, overlap))
    out: List[str] = []
    # precompute min to avoid repeated bound checks in the loop
    for start in range(0, n, step):
        end = start + chunk_size
        if end >= n:
            end = n
        piece = enc.decode(toks[start:end]).strip()
        if piece:
            out.append(piece)
        if end == n:
            break
    return tuple(out)

def chunk_text(
    text: str,
    chunk_size: int = KB_CHUNK_SIZE,
    overlap: int = KB_CHUNK_OVERLAP
) -> List[str]:
    # thin wrapper to return list (public API) while keeping cache hit on core work
    return list(_chunk_text_cached(text or "", int(chunk_size), int(overlap)))

In [ ]:
# Step 4: File parsing — optimized + cached

from pypdf import PdfReader
from docx import Document
from bs4 import BeautifulSoup
import html2text
import io, os
from functools import lru_cache
from typing import List

# allow-listed extensions
ALLOWED_EXTS = {".txt", ".md", ".pdf", ".docx", ".html", ".htm"}

@lru_cache(maxsize=256)
def _norm_ext(path: str) -> str:
    return os.path.splitext((path or "").lower())[1]

@lru_cache(maxsize=1024)
def _is_allowed(path: str) -> bool:
    return _norm_ext(path) in ALLOWED_EXTS

# one-time html2text config (cheaper than making a new object per call)
@lru_cache(maxsize=1)
def _h2t():
    h = html2text.HTML2Text()
    h.ignore_images = True
    h.ignore_links = False  # keep links for context
    h.ignore_tables = True
    h.body_width = 0        # no line wrapping; preserves text
    return h

# small helper to build a lightweight cache key for bytes without storing the bytes
def _bytes_cache_key(filename: str, data: bytes) -> tuple:
    # use length + a short hash of first/last 8KB to avoid full hashing cost
    if not data:
        return (filename.lower(), 0, 0, 0)
    head = data[:8192]
    tail = data[-8192:] if len(data) > 8192 else b""
    # simple, fast rolling hash
    h = 1469598103934665603
    for b in head:
        h = (h ^ b) * 1099511628211
    for b in tail:
        h = (h ^ b) * 1099511628211
    return (filename.lower(), len(data), h & 0xFFFFFFFF)

@lru_cache(maxsize=128)
def _extract_text_from_bytes_cached(key: tuple) -> str:
    """
    Internal cached worker. The real parser below supplies (filename,len,hash) as key
    and keeps the raw bytes out of the cache.
    """
    # This function is only called via the public wrapper; it never sees raw bytes.
    # We stash the actual parsing logic in the wrapper and duplicate just the
    # branching here for cache correctness clarity.
    # (We still do the real parse in the wrapper to avoid passing large bytes into cache.)
    return ""  # placeholder; never executed

def _parse_pdf_bytes(data: bytes) -> str:
    reader = PdfReader(io.BytesIO(data or b""))
    parts: List[str] = []
    for p in reader.pages:
        try:
            parts.append(p.extract_text() or "")
        except Exception:
            parts.append("")
    return clean_text("\n".join(parts))

def _parse_docx_bytes(data: bytes) -> str:
    doc = Document(io.BytesIO(data or b""))
    return clean_text("\n".join((p.text or "") for p in doc.paragraphs))

def _parse_html_bytes(data: bytes) -> str:
    soup = BeautifulSoup(data or b"", "html.parser")
    main = soup.find(["article", "main"]) or soup.body or soup
    return clean_text(_h2t().handle(str(main)))

def _parse_text_bytes(data: bytes) -> str:
    return clean_text((data or b"").decode("utf-8", errors="ignore"))

def _extract_text_from_bytes(filename: str, data: bytes) -> str:
    """
    Return plain text from supported files. Fast paths + caching.
    """
    name = (filename or "").lower()
    key = _bytes_cache_key(name, data)

    # quick manual cache check to avoid storing raw bytes in lru_cache
    # we piggyback on the cached stub by storing results in a tiny side-map
    # keyed by (filename,len,rolling_hash).
    if not hasattr(_extract_text_from_bytes, "_memo"):
        _extract_text_from_bytes._memo = {}
    memo = _extract_text_from_bytes._memo  # type: ignore[attr-defined]
    if key in memo:
        return memo[key]

    try:
        if name.endswith(".pdf"):
            out = _parse_pdf_bytes(data)
        elif name.endswith(".docx"):
            out = _parse_docx_bytes(data)
        elif name.endswith((".html", ".htm")):
            out = _parse_html_bytes(data)
        else:  # .txt / .md or unknown → treat as text
            out = _parse_text_bytes(data)
    except Exception as e:
        out = f"[parse error: {e}]"

    # memoize compactly (avoid storing big blobs)
    if len(memo) > 128:  # simple bound to avoid unbounded growth
        # drop ~25% oldest entries
        for i, k in enumerate(list(memo.keys())):
            if i % 4 == 0:
                memo.pop(k, None)
    memo[key] = out
    return out

In [ ]:
# Step 5: Shared-link downloading & basic file IO — optimized + cached

import os, io, zipfile, tempfile, urllib.parse, requests, time, hashlib
from typing import List
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd  # used by the Excel loader below

# small on-disk cache for Dropbox zips (persists across cells)
_DBX_CACHE_DIR = "/content/.dbx_cache"
os.makedirs(_DBX_CACHE_DIR, exist_ok=True)

def _force_dl(url: str) -> str:
    """Make a Dropbox share link a direct-download URL."""
    u = urllib.parse.urlparse(url or "")
    q = urllib.parse.parse_qs(u.query)
    q["dl"] = ["1"]
    new_q = urllib.parse.urlencode({k: v[0] for k, v in q.items()})
    return urllib.parse.urlunparse((u.scheme, u.netloc, u.path, u.params, new_q, u.fragment))

def _safe_cache_name(url: str) -> str:
    """Stable filename for the url content."""
    h = hashlib.sha1(url.encode("utf-8")).hexdigest()
    return os.path.join(_DBX_CACHE_DIR, f"{h}.zip")

def _requests_session() -> requests.Session:
    """Session with retries on transient errors."""
    s = requests.Session()
    retries = Retry(
        total=4,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "POST"])
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.mount("http://", HTTPAdapter(max_retries=retries))
    s.headers.update({"User-Agent": "RAGBot/1.0 (+colab)"})
    return s

def _download_zip_if_needed(dl_url: str, cache_path: str, max_age_sec: int = 24*3600) -> str:
    """Download the zip to cache_path if missing or stale. Return cache path."""
    if os.path.exists(cache_path):
        age = time.time() - os.path.getmtime(cache_path)
        if age < max_age_sec and os.path.getsize(cache_path) > 0:
            return cache_path

    sess = _requests_session()
    with sess.get(dl_url, stream=True, timeout=120) as r:
        r.raise_for_status()
        tmp_path = cache_path + ".part"
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        os.replace(tmp_path, cache_path)
    return cache_path

def _download_and_extract_shared_folder(url: str) -> str:
    """
    Download a Dropbox shared folder as zip (cached), extract to a temp dir,
    and return the extract path. Extracts allowed text/doc files AND Excel.
    Skips macOS metadata.
    """
    if not url:
        raise ValueError("shared URL is empty")

    dl_url = _force_dl(url)
    cache_zip = _safe_cache_name(dl_url)
    zip_path = _download_zip_if_needed(dl_url, cache_zip)

    extract_dir = tempfile.mkdtemp(prefix="dbx_extracted_")
    with zipfile.ZipFile(zip_path, "r") as z:
        excel_exts = (".xlsx", ".xls")
        members = []
        for info in z.infolist():
            if info.is_dir():
                continue
            fn = info.filename
            # skip macOS junk
            if "/__MACOSX/" in f"/{fn}/" or os.path.basename(fn).startswith("._"):
                continue
            # extract if indexable or excel (so we can load trusted sources)
            if _is_allowed(fn) or fn.lower().endswith(excel_exts):
                members.append(info)

        # fallback: if nothing matched, extract all (still skip macOS junk)
        if not members:
            for info in z.infolist():
                if info.is_dir():
                    continue
                fn = info.filename
                if "/__MACOSX/" in f"/{fn}/" or os.path.basename(fn).startswith("._"):
                    continue
                z.extract(info, extract_dir)
        else:
            for info in members:
                z.extract(info, extract_dir)
    return extract_dir

def _walk_local_allowed(root_dir: str) -> List[str]:
    """Collect allowed files under root_dir (skip macOS metadata)."""
    paths: List[str] = []
    for dirpath, _, filenames in os.walk(root_dir):
        if "__MACOSX" in dirpath:
            continue
        for fn in filenames:
            if fn.startswith("._"):
                continue
            p = os.path.join(dirpath, fn)
            if _is_allowed(p):
                paths.append(p)
    return paths

def _read_local_file_bytes(path: str) -> bytes:
    """Read file as bytes."""
    with open(path, "rb") as f:
        return f.read()

# Trusted sources helpers (Excel)

# Use existing globals
TRUSTED_SOURCES     = globals().get("TRUSTED_SOURCES", {"names": [], "domains": []})
EXTERNAL_EXCEL_PATH = globals().get("EXTERNAL_EXCEL_PATH", "/content/Chatbot Trusted Sources.xlsx")
EXTERNAL_SHARED_URL = globals().get("EXTERNAL_SHARED_URL", "")

def _normalize_domain_for_excel(s: str) -> str:
    """Normalize URLs/hosts to a bare domain like 'example.com'."""
    try:
        s = str(s or "").strip()
        if not s:
            return ""
        from urllib.parse import urlparse
        netloc = urlparse(s if s.startswith(("http://","https://")) else f"https://{s}").netloc.lower()
        return netloc[4:] if netloc.startswith("www.") else netloc
    except Exception:
        return ""

def _extract_domains_from_cell_excel(val) -> list[str]:
    """Find domains in a free-text Excel cell."""
    import re
    text = str(val or "")
    hits = re.findall(r"(https?://[^\s]+|(?:[A-Za-z0-9-]+\.)+[A-Za-z]{2,24})", text)
    out = set()
    for h in hits + [text]:
        dom = _normalize_domain_for_excel(h)
        if dom and "." in dom and " " not in dom:
            out.add(dom)
    return sorted(out)

def load_trusted_sources_from_excel(path: str) -> dict:
    """Read Excel of trusted sources -> {'names': [...], 'domains': [...]}."""
    out = {"names": [], "domains": []}
    try:
        if not os.path.exists(path):
            print(f"⚠️ Excel not found at {path}.")
            return out

        df = pd.read_excel(path, engine="openpyxl")
        if df.empty:
            return out

        cols = {c: str(c).lower() for c in df.columns}
        name_cols = [c for c,l in cols.items() if any(k in l for k in ("name","source","author","publication","journal","outlet"))]
        url_cols  = [c for c,l in cols.items() if any(k in l for k in ("website","url","domain","link"))]

        names = []
        domains = set()

        for _, row in df.iterrows():
            for c in name_cols:
                val = str(row.get(c, "")).strip()
                if val:
                    names.append(val)

            if url_cols:
                for c in url_cols:
                    domains.update(_extract_domains_from_cell_excel(row.get(c, "")))
            else:
                for v in row.values:
                    domains.update(_extract_domains_from_cell_excel(v))

        out["names"] = [n for n in (n.strip() for n in names) if n]
        out["domains"] = sorted({d for d in domains if "." in d and " " not in d})
        return out

    except Exception as e:
        print(f"⚠️ Could not read trusted sources Excel: {e}")
        return out

def _ensure_excel_present():
    """If the Excel was uploaded to /mnt/data, copy into the expected path."""
    try:
        src = "/mnt/data/Chatbot Trusted Sources.xlsx"
        if (not os.path.exists(EXTERNAL_EXCEL_PATH)) and os.path.exists(src):
            import shutil
            os.makedirs(os.path.dirname(EXTERNAL_EXCEL_PATH), exist_ok=True)
            shutil.copy2(src, EXTERNAL_EXCEL_PATH)
            print(f"Copied trusted sources Excel to {EXTERNAL_EXCEL_PATH}")
    except Exception as e:
        print(f"⚠️ ensure_excel_present: {e}")

def _load_trusted_sources():
    """Populate TRUSTED_SOURCES from Excel if present; keep empty if not."""
    global TRUSTED_SOURCES
    _ensure_excel_present()
    ts = load_trusted_sources_from_excel(EXTERNAL_EXCEL_PATH)
    if ts["names"] or ts["domains"]:
        TRUSTED_SOURCES = ts
        print(f"Trusted sources loaded — names: {len(ts['names'])}, domains: {len(ts['domains'])}")
        if ts["domains"]:
            print("  sample domains:", ts["domains"][:8])
    else:
        TRUSTED_SOURCES = {"names": [], "domains": []}
        print("ℹ️ No trusted sources found; proceeding without trusted web domains.")

def ensure_trusted_sources_from_external_folder():
    """Try to fetch an Excel from the external Dropbox share."""
    try:
        if not EXTERNAL_SHARED_URL:
            return
        local_dir = _download_and_extract_shared_folder(EXTERNAL_SHARED_URL)
        excel_path = None
        for root, _, files in os.walk(local_dir):
            for fn in files:
                if fn.lower().endswith((".xlsx",".xls")):
                    excel_path = os.path.join(root, fn)
                    break
            if excel_path:
                break

        if not excel_path:
            print("⚠️ No Excel found in external folder.")
            return

        import shutil
        os.makedirs(os.path.dirname(EXTERNAL_EXCEL_PATH), exist_ok=True)
        shutil.copy2(excel_path, EXTERNAL_EXCEL_PATH)
        _load_trusted_sources()

    except Exception as e:
        print(f"⚠️ ensure_trusted_sources_from_external_folder: {e}")

In [ ]:
# Step 6: Embeddings — cached & resilient

from typing import List, Tuple
import time, random, hashlib

def _norm(s: str) -> str:
    return (s or "").strip()

def _doc_cache_key(text: str) -> str:
    # compact, stable key for doc chunks
    return hashlib.sha1(_norm(text).encode("utf-8")).hexdigest()

def embed_texts(
    texts: List[str],
    batch: int = 128,
    max_retries: int = 4,
    use_cache: bool = True,
) -> List[List[float]]:
    """
    Embed list[str] with:
      - cache hits served from _EMBED_CACHE_DOC
      - only cache misses sent to OpenAI in batches
      - exponential backoff on transient errors
    Returns vectors aligned 1:1 with `texts`.
    """
    if not texts:
        return []

    # Prepare cache lookups
    keys = [_doc_cache_key(t) for t in texts]
    out: List[List[float]] = [None] * len(texts)  # type: ignore
    to_embed: List[Tuple[int, str, str]] = []     # (idx, text, key)

    for i, (t, k) in enumerate(zip(texts, keys)):
        if use_cache and k in _EMBED_CACHE_DOC:
            out[i] = _EMBED_CACHE_DOC[k]
        else:
            nt = _norm(t)
            if nt:
                to_embed.append((i, nt, k))
            else:
                out[i] = []  # empty input → empty vec

    # Batch-embed only misses
    pos = 0
    while pos < len(to_embed):
        chunk = to_embed[pos:pos+batch]
        pos += batch

        idxs = [i for (i, _t, _k) in chunk]
        payload = [_t for (_i, _t, _k) in chunk]
        keys_slice = [_k for (_i, _t, _k) in chunk]

        for attempt in range(max_retries):
            try:
                resp = client.embeddings.create(model=EMBED_MODEL, input=payload)
                vecs = [d.embedding for d in resp.data]
                for j, vec in enumerate(vecs):
                    dst = idxs[j]
                    k = keys_slice[j]
                    out[dst] = vec
                    if use_cache:
                        _EMBED_CACHE_DOC[k] = vec
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    raise
                sleep_s = (0.6 * (2 ** attempt)) + random.uniform(0, 0.25)
                _dbg(f"embed_texts retry {attempt+1}: {e} → sleep {sleep_s:.2f}s")
                time.sleep(sleep_s)

    # At this point all positions should be filled
    return out  # type: ignore

def _embed_query(
    text: str,
    max_retries: int = 4,
    use_cache: bool = True,
) -> List[float]:
    """
    Embed a single query string with caching in _EMBED_CACHE_Q.
    """
    q = _norm(text)
    if use_cache and q in _EMBED_CACHE_Q:
        return _EMBED_CACHE_Q[q]

    for attempt in range(max_retries):
        try:
            resp = client.embeddings.create(model=EMBED_MODEL, input=[q])
            vec = resp.data[0].embedding
            if use_cache:
                _EMBED_CACHE_Q[q] = vec
            return vec
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            sleep_s = (0.6 * (2 ** attempt)) + random.uniform(0, 0.25)
            _dbg(f"_embed_query retry {attempt+1}: {e} → sleep {sleep_s:.2f}s")
            time.sleep(sleep_s)

    return []

In [ ]:
# Step 7: Indexing (Dropbox → text → chunks → embeddings → Chroma)

import json, hashlib, os, shutil

MANIFEST_PATH = "/content/dbx_manifest.json"

def load_manifest() -> dict:
    try:
        with open(MANIFEST_PATH, "r") as f:
            return json.load(f)
    except Exception:
        return {}

def save_manifest(m: dict) -> None:
    with open(MANIFEST_PATH, "w") as f:
        json.dump(m, f)

MANIFEST = load_manifest()

def file_sig(path: str, data: bytes) -> str:
    return hashlib.sha1(path.encode() + b"::" + data).hexdigest()

def _manifest_key(shared_url: str, rel_path: str, tag: str) -> str:
    return f"{tag}::{shared_url}::{rel_path}"

def _should_reembed(shared_url: str, rel_path: str, tag: str, data: bytes) -> bool:
    key = _manifest_key(shared_url, rel_path, tag)
    sig = file_sig(rel_path, data)
    return MANIFEST.get(key) != sig

def _mark_embedded(shared_url: str, rel_path: str, tag: str, data: bytes) -> None:
    key = _manifest_key(shared_url, rel_path, tag)
    MANIFEST[key] = file_sig(rel_path, data)

def index_shared_link_folder(shared_url: str, collection, tag: str):
    print(f"Indexing → {collection.name} ({tag})")
    local_dir = _download_and_extract_shared_folder(shared_url)
    try:
        local_paths = _walk_local_allowed(local_dir)
        if not local_paths:
            print("  no eligible files")
            return

        try:
            existing = collection.get(ids=None, where={"tag": tag})
            already = set(existing.get("ids", []))
        except Exception:
            already = set()

        docs_to_add, ids_to_add, metas_to_add = [], [], []
        skipped_unchanged = 0

        for p in local_paths:
            try:
                raw = _read_local_file_bytes(p)
                rel = p.replace(local_dir, "").lstrip(os.sep)
            except Exception as e:
                print(f"  skip read error: {p} → {e}")
                continue

            if not _should_reembed(shared_url, rel, tag, raw):
                skipped_unchanged += 1
                _mark_embedded(shared_url, rel, tag, raw)
                continue

            text = _extract_text_from_bytes(p, raw)
            if not text or (text.startswith("[") and "parse error" in text.lower()):
                print(f"  skip parse error: {rel}")
                _mark_embedded(shared_url, rel, tag, raw)
                continue

            chunks = [c for c in (chunk_text(text) or []) if c.strip()]
            if not chunks:
                _mark_embedded(shared_url, rel, tag, raw)
                continue

            for idx, ch in enumerate(chunks):
                uid = sha1(f"{shared_url}::{rel}::{idx}")
                if uid in already:
                    continue
                docs_to_add.append(ch)
                ids_to_add.append(uid)
                metas_to_add.append({
                    "source_path": rel,
                    "shared_link": shared_url,
                    "chunk_index": idx,
                    "tag": tag
                })

            _mark_embedded(shared_url, rel, tag, raw)

        if skipped_unchanged:
            print(f"  skipped unchanged files: {skipped_unchanged}")

        if not docs_to_add:
            print("  nothing new")
            save_manifest(MANIFEST)
            return

        # SPEED-UP: de-dupe chunk texts within this run before embedding
        # global _EMBED_CACHE_DOC helps across runs, avoids duplicates in this call
        key_for = lambda s: hashlib.sha1(s.encode("utf-8")).hexdigest()
        uniq_keys = []
        uniq_texts = []
        seen = set()
        key_list = []

        for ch in docs_to_add:
            k = key_for(ch)
            key_list.append(k)
            if k not in seen:
                seen.add(k)
                uniq_keys.append(k)
                uniq_texts.append(ch)

        print(f"  embedding chunks: {len(docs_to_add)} (unique: {len(uniq_texts)})")
        uniq_vecs = embed_texts(uniq_texts)  # this still uses batching + global cache

        # map unique vecs back to full order
        vec_map = {k: v for k, v in zip(uniq_keys, uniq_vecs)}
        vecs = [vec_map[k] for k in key_list]

        print("  adding to chroma…")
        collection.add(
            documents=docs_to_add,
            embeddings=vecs,
            metadatas=metas_to_add,
            ids=ids_to_add
        )
        print(f"  done. collection size: {collection.count()}")
        save_manifest(MANIFEST)

    finally:
        try:
            shutil.rmtree(local_dir, ignore_errors=True)
        except Exception:
            pass

def build_or_update_indexes():
    did = False

    if KB_SHARED_URL:
        index_shared_link_folder(KB_SHARED_URL, kb_collection, tag="kb")
        did = True

    if STYLE_SHARED_URL:
        index_shared_link_folder(STYLE_SHARED_URL, style_collection, tag="style")
        did = True

    if EXTERNAL_SHARED_URL:
        index_shared_link_folder(EXTERNAL_SHARED_URL, external_collection, tag="external")
        # load trusted sources (excel local or from external dropbox if needed)
        _load_trusted_sources()
        if not (TRUSTED_SOURCES.get("names") or TRUSTED_SOURCES.get("domains")):
            ensure_trusted_sources_from_external_folder()

        dom_ct = len(TRUSTED_SOURCES.get("domains", []))
        print(f"trusted web domains: {dom_ct}")
        if dom_ct:
            print("  examples:", TRUSTED_SOURCES["domains"][:8])
        did = True

    if not did:
        print("⚠️ no shared URLs configured")

In [ ]:
# Step 8: Retrieval (query → vectors → nearest chunks) — patched for KB no-limit

# tiny in-session cache for retrieval results
_RETR_CACHE: dict[tuple, list[tuple[str, dict, str]]] = globals().setdefault("_RETR_CACHE", {})

def _embed_query(text: str) -> List[float]:
    """Embed a single query string with cache."""
    key = (text or "").strip()
    cached = _EMBED_CACHE_Q.get(key)
    if cached is not None:
        return cached
    resp = client.embeddings.create(model=EMBED_MODEL, input=[key])
    vec = resp.data[0].embedding
    _EMBED_CACHE_Q[key] = vec
    return vec

def _unpack(res: Dict[str, Any]) -> Tuple[List[str], List[dict], List[str]]:
    """Normalize chroma query result."""
    docs  = res.get("documents", [[]])[0] if res.get("documents") else []
    metas = res.get("metadatas", [[]])[0] if res.get("metadatas") else []
    ids   = res.get("ids", [[]])[0]       if res.get("ids")       else []
    if not ids or len(ids) != len(docs):
        ids = [None] * len(docs)
    return docs, metas, ids

def _safe_count(collection) -> int:
    """Safely get a collection's count, returning 0 on error."""
    try:
        c = collection.count()
        return int(c) if c is not None else 0
    except Exception:
        return 0

def _query_collection(collection, qvec: List[float], top_k: Optional[int], tag: str | None = None):
    """
    Shared helper to query a collection with sane defaults + guards.
    If top_k is None, return all available matches.
    """
    max_k = _safe_count(collection)
    if max_k < 1:
        return []

    n = max_k if top_k is None else max(1, min(int(top_k), max_k))
    where = {"tag": tag} if tag else None

    res = collection.query(
        query_embeddings=[qvec],
        n_results=n,
        include=["documents", "metadatas"],
        where=where
    )
    docs, metas, ids = _unpack(res)
    return list(zip(docs, metas, ids))

def retrieve_kb(question: str, top_k: Optional[int] = KB_TOP_K) -> List[Tuple[str, Dict[str, Any], Any]]:
    key = (kb_collection.name, question, top_k if top_k is not None else "ALL")
    if key in _RETR_CACHE:
        return _RETR_CACHE[key]
    qvec = _embed_query(question)
    out = _query_collection(kb_collection, qvec, top_k, tag="kb")
    _RETR_CACHE[key] = out
    return out

def retrieve_style(question: str, top_k: int = STYLE_TOP_K) -> List[Tuple[str, Dict[str, Any], Any]]:
    key = (style_collection.name, question, int(top_k))
    if key in _RETR_CACHE:
        return _RETR_CACHE[key]
    qvec = _embed_query(question)
    out = _query_collection(style_collection, qvec, top_k, tag="style")
    _RETR_CACHE[key] = out
    return out

def retrieve_external(question: str, top_k: int = 4) -> List[Tuple[str, Dict[str, Any], Any]]:
    key = (external_collection.name, question, int(top_k))
    if key in _RETR_CACHE:
        return _RETR_CACHE[key]
    qvec = _embed_query(question)
    out = _query_collection(external_collection, qvec, top_k, tag="external")
    _RETR_CACHE[key] = out
    return out

In [ ]:
# Step 9: Prompting & guardrails — updated for W#/P# buckets and K-first weighting

import re as _re

# prompt-size caps
_MAX_W_SNIPPET_CHARS = 900
_MAX_P_SNIPPET_CHARS = 900
_MAX_K_SNIPPET_CHARS = 900
_MAX_STYLE_SNIPPET_CHARS = 600

# tiny cache for composed messages
_MSG_CACHE: dict[tuple, list[dict]] = globals().setdefault("_MSG_CACHE", {})

SYSTEM_PROMPT = """ you are a helpful assistant answering questions from HR executives who are members of an exclusive network called HRNXT. Answer using ONLY the material provided below, in this strict order of priority:
1) [K#] Knowledge Snippets (internal library)
2) [W#] Trusted Web Passages (from allowed domains)
3) [P#] People-authored pages (articles/books by named authors)
4) [G:SourceName] Model background about a named trusted source — only if no usable [W#]/[P#] exists.

Rules:
- Do NOT use the open web beyond [W#] and [P#]. Do not invent citations.
- Cite inline with [K#], [W#], [P#], and [G:SourceName] (background only).
- If context is insufficient, say so and ask for more docs, domains, or author names.
- Style snippets affect tone only; never import facts from style.
"""

def _format_sources_map(label: str, lst: list[tuple[str, dict, str]]) -> str:
    lines = []
    for i, (_, meta, _id) in enumerate(lst, start=1):
        src  = meta.get("source_path", "unknown")
        link = meta.get("shared_link")
        lines.append(f"[{label}{i}] {src}" + (f"  (from: {link})" if link else ""))
    return "\n".join(lines) if lines else "(none)"

def _truncate(s: str, lim: int) -> str:
    s = (s or "").strip()
    if len(s) <= lim:
        return s
    return s[:lim].rstrip() + " …"

def _compose_cache_key(
    question: str,
    kb_hits, style_hits, web_domain_snippets, web_people_snippets
) -> tuple:
    kb_ids = tuple((t[2] or f"{t[1].get('source_path','')}#{t[1].get('chunk_index','')}") for t in kb_hits)
    st_ids = tuple((t[2] or f"{t[1].get('source_path','')}#{t[1].get('chunk_index','')}") for t in style_hits)
    w_urls = tuple((w.get("url") or "") for w in (web_domain_snippets or []))
    p_urls = tuple((w.get("url") or "") for w in (web_people_snippets or []))
    return (question or "", kb_ids, st_ids, w_urls, p_urls)

def compose_messages(
    question: str,
    kb_hits: List[Tuple[str, Dict[str,Any], str]],
    style_hits: List[Tuple[str, Dict[str,Any], str]],
    web_domain_snippets: List[Dict[str, Any]],
    web_people_snippets: List[Dict[str, Any]],
):
    _key = _compose_cache_key(question, kb_hits, style_hits, web_domain_snippets, web_people_snippets)
    if _key in _MSG_CACHE:
        return _MSG_CACHE[_key]

    # KB blocks
    kb_blocks = [
        f"[K{i}] {_truncate(doc, _MAX_K_SNIPPET_CHARS)}"
        for i, (doc, _, _) in enumerate(kb_hits, start=1)
        if (doc or "").strip()
    ]
    kb_text    = "\n\n".join(kb_blocks) if kb_blocks else "(no knowledge snippets)"
    kb_sources = _format_sources_map("K", kb_hits)

    # W# domain blocks
    w_blocks, w_sources = [], []
    for i, w in enumerate(web_domain_snippets or [], start=1):
        body   = (w.get("content") or "").strip()
        if not body:
            continue
        title  = (w.get("title") or w.get("url") or "").strip()
        url    = w.get("url", "")
        domain = (w.get("domain") or "").strip()
        snippet = _truncate(body, _MAX_W_SNIPPET_CHARS)
        w_blocks.append(f"[W{i}] {snippet}")
        w_sources.append(f"[W{i}] {title}  ({domain})  {url}")
    w_text = "\n\n".join(w_blocks) if w_blocks else "(no trusted web snippets)"
    w_map  = "\n".join(w_sources) if w_sources else "(none)"

    # P# people-authored blocks
    p_blocks, p_sources = [], []
    for i, w in enumerate(web_people_snippets or [], start=1):
        body   = (w.get("content") or "").strip()
        if not body:
            continue
        title  = (w.get("title") or w.get("url") or "").strip()
        url    = w.get("url", "")
        domain = (w.get("domain") or "").strip()
        snippet = _truncate(body, _MAX_P_SNIPPET_CHARS)
        p_blocks.append(f"[P{i}] {snippet}")
        p_sources.append(f"[P{i}] {title}  ({domain})  {url}")
    p_text = "\n\n".join(p_blocks) if p_blocks else "(no people-authored pages)"
    p_map  = "\n".join(p_sources) if p_sources else "(none)"

    # Style (tone only)
    style_samples = []
    for (doc, _, _) in style_hits[:STYLE_TOP_K]:
        if (doc or "").strip():
            style_samples.append(_truncate(doc, _MAX_STYLE_SNIPPET_CHARS))
    style_text = "\n\n---\n\n".join(style_samples) if style_samples else "(no style examples)"

    # Guidance
    facts_blob = "\n".join([kb_text, w_text, p_text])
    facts_tokens = count_tokens(facts_blob)
    if facts_tokens >= MIN_CONTEXT_TOKENS:
        guidance = ("Prioritize [K#], then [W#], then [P#]. "
                    "Only if no [W#]/[P#] supports a named source may you use [G:SourceName].")
    else:
        guidance = ("Context is thin. Answer briefly, note gaps, and request more docs/domains/authors. "
                    "You may reference [G:SourceName] sparingly only if no [W#]/[P#] exists for that source.")

    # Final user block
    user_block = f"""Question:
{question}

Knowledge Snippets ([K#]):
{kb_text}

Trusted Web Passages ([W#]):
{w_text}

People-authored Pages ([P#]):
{p_text}

Style Excerpts (tone only):
{style_text}

Source Maps
Knowledge:
{kb_sources}

Web (domains):
{w_map}

Web (people):
{p_map}

Instructions:
- {guidance}
- Use inline citations [K#], [W#], [P#], and/or [G:SourceName] (background only).
- Be clear and concise unless depth is requested.
"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_block},
    ]
    _MSG_CACHE[_key] = messages
    return messages

In [ ]:
# Step 10: Chat loop + helpers — printout updated for W#/P#

import time, hashlib, json
from typing import Dict, Any, List, Tuple, Optional

# tiny answer cache and run guard
_ANSWER_CACHE: dict[tuple, Dict[str, Any]] = globals().setdefault("_ANSWER_CACHE", {})
_INDEX_BUILT = globals().setdefault("_INDEX_BUILT", False)

# Domain-aware cache key helpers

def _current_domains() -> List[str]:
    try:
        return [d.strip().lower() for d in (TRUSTED_SOURCES.get("domains") or []) if isinstance(d, str) and d.strip()]
    except Exception:
        return []

def _domains_signature(domains: List[str]) -> str:
    try:
        norm = sorted(set(d.strip().lower() for d in (domains or []) if isinstance(d, str) and d.strip()))
        payload = {"domains": norm, "serper": bool(SERPER_API_KEY)}
        return hashlib.sha1(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:12]
    except Exception:
        return f"len{len(domains or [])}_serper{bool(SERPER_API_KEY)}"

def _answer_cache_key(q: str) -> tuple:
    qn = (q or "").strip().lower()
    sig = _domains_signature(_current_domains())
    return (qn, sig)

_LAST_DOMAIN_SIG = globals().get("_LAST_DOMAIN_SIG", None)

def _maybe_reload_trusted_sources():
    global _LAST_DOMAIN_SIG, _ANSWER_CACHE

    if not _current_domains():
        _load_trusted_sources()
        if not _current_domains():
            ensure_trusted_sources_from_external_folder()

    sig = _domains_signature(_current_domains())
    if sig != _LAST_DOMAIN_SIG:
        _ANSWER_CACHE.clear()
        _LAST_DOMAIN_SIG = sig

def _print_citations(out: Dict[str, Any]) -> None:
    print("\n— cited from —")
    shown = False

    if out.get("kb_hits"):
        print("Knowledge:")
        for i, (_, meta, _) in enumerate(out["kb_hits"], start=1):
            sp = meta.get("source_path", "unknown")
            sl = meta.get("shared_link")
            print(f"[K{i}] {sp}" + (f"  (from: {sl})" if sl else ""))
            shown = True

    if out.get("web_domain_snippets"):
        print("Trusted web (domains):")
        for i, w in enumerate(out["web_domain_snippets"], start=1):
            title = (w.get("title") or "").strip()
            url   = w.get("url", "")
            dom   = (w.get("domain") or "").strip()
            label = title if title else url
            print(f"[W{i}] {label}  ({dom})  {url}")
            shown = True

    if out.get("web_people_snippets"):
        print("People-authored (web):")
        for i, w in enumerate(out["web_people_snippets"], start=1):
            title = (w.get("title") or "").strip()
            url   = w.get("url", "")
            dom   = (w.get("domain") or "").strip()
            label = title if title else url
            print(f"[P{i}] {label}  ({dom})  {url}")
            shown = True

    if not shown:
        print("(no snippets found)")

def _ensure_indexed_once():
    global _INDEX_BUILT
    if _INDEX_BUILT:
        return
    print("📥 indexing your Dropbox folders (KB / Style / External / Trusted Web)…")
    t0 = time.perf_counter()
    build_or_update_indexes()
    _INDEX_BUILT = True
    print(f"✅ ready in {time.perf_counter() - t0:.2f}s. type your question (or 'exit').\n")

def _get_or_generate(q: str) -> Dict[str, Any]:
    key = _answer_cache_key(q)
    if key in _ANSWER_CACHE:
        return _ANSWER_CACHE[key]
    out = generate_answer(q)
    _ANSWER_CACHE[key] = out
    if len(_ANSWER_CACHE) > 30:
        _ANSWER_CACHE.pop(next(iter(_ANSWER_CACHE)))
    return out

def chat():
    _ensure_indexed_once()
    while True:
        try:
            q = input("You: ").strip()
        except EOFError:
            break
        if not q:
            continue
        if q.lower() in {"exit", "quit", "q"}:
            print("Bye!")
            break

        _maybe_reload_trusted_sources()

        t0 = time.perf_counter()
        try:
            out = _get_or_generate(q)
        except Exception as e:
            print(f"\nAssistant:\nSorry, something went wrong: {e}")
            continue

        print("\nAssistant:\n")
        print(out["answer"])
        _print_citations(out)
        print(f"\n⏱️ answered in {time.perf_counter() - t0:.2f}s")

def ask(q: str):
    _ensure_indexed_once()
    _maybe_reload_trusted_sources()
    t0 = time.perf_counter()
    out = _get_or_generate(q)
    print("\nAssistant:\n")
    print(out["answer"])
    _print_citations(out)
    print(f"\n⏱️ answered in {time.perf_counter() - t0:.2f}s")

In [ ]:
# Step 11: Answer generation — tighten fetching & cap total web snippets to 5

from typing import Dict, Any, List, Tuple
import time, re
from urllib.parse import urlparse

_WEB_DEBUG = True  # set False to silence debug logs

def _log(msg: str):
    if _WEB_DEBUG:
        print(f"[WEB] {msg}")

# Trusted Web helpers (Serper + direct fetch fallback)

_SKIP_EXTS = (".pdf", ".xml", ".rss", ".atom")

def _looks_heavy(url: str) -> bool:
    u = url.lower()
    return any(u.endswith(ext) or f"{ext}?" in u for ext in _SKIP_EXTS)

def _is_allowed_domain(url: str, allowed: List[str]) -> bool:
    try:
        host = urlparse(url).netloc.lower()
        if host.startswith("www."):
            host = host[4:]
        return any(host == d or host.endswith("." + d) for d in allowed)
    except Exception:
        return False

def _serper_search_raw(query: str, num: int = 8) -> List[Dict[str, Any]]:
    if not SERPER_API_KEY:
        _log("SERPER_API_KEY missing; skipping Serper.")
        return []
    cache_key = (query.strip(), int(num))
    if cache_key in _SERPER_CACHE:
        _log(f"Serper cache hit for: {query!r}")
        return _SERPER_CACHE[cache_key]
    try:
        r = requests.post(
            "https://google.serper.dev/search",
            headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
            json={"q": query, "num": int(num), "gl": "us", "hl": "en"},
            timeout=15,
        )
        if r.status_code != 200:
            _log(f"Serper non-200: {r.status_code} → {(r.text or '')[:160]}")
            return []
        data = r.json() or {}
        items = []
        for it in (data.get("organic") or []):
            url = (it.get("link") or "").strip()
            if not url:
                continue
            # prefilter heavy content
            if _looks_heavy(url):
                continue
            items.append({"title": (it.get("title") or "").strip(), "url": url})
        _SERPER_CACHE[cache_key] = items
        return items
    except Exception as e:
        _log(f"Serper error: {e}")
        return []

def _serper_search_domains(question: str, allowed_domains: List[str], num: int) -> List[Dict[str, Any]]:
    if not allowed_domains:
        return []
    site_clause = " OR ".join([f"site:{d}" for d in allowed_domains[:12]])
    query = f"{question} {site_clause}".strip()
    hits = _serper_search_raw(query, num=min(num, 12))
    kept = []
    for it in hits:
        if _is_allowed_domain(it["url"], allowed_domains):
            kept.append(it)
    return kept

def _http_get(url: str, timeout: int = 15) -> str:
    # cache even non-HTML as empty string sentinel to avoid repeated downloads
    if url in _HTTP_CACHE:
        return _HTTP_CACHE[url]
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent": "RAGBot/1.0 (+colab)"})
        ctype = (r.headers.get("Content-Type","") or "").lower()
        if r.status_code == 200 and "text/html" in ctype:
            _HTTP_CACHE[url] = r.text
            return r.text
        _HTTP_CACHE[url] = ""  # sentinel: don't re-fetch this non-HTML URL again
        _log(f"SKIP non-HTML {url} → {r.status_code} {ctype}")
    except Exception as e:
        _HTTP_CACHE[url] = ""  # sentinel
        _log(f"http get failed: {e} {url}")
    return ""

def _extract_readable_text(html: str) -> str:
    if not html:
        return ""
    try:
        soup = BeautifulSoup(html, "html.parser")
        main = soup.find(["article", "main"]) or soup.body or soup
        return clean_text(_h2t().handle(str(main)))
    except Exception as e:
        _log(f"extract error: {e}")
        return ""

def _dedupe_by_domain(items: List[Dict[str, Any]], per_domain: int = 1) -> List[Dict[str, Any]]:
    bucket: Dict[str, int] = {}
    out = []
    for it in items:
        host = urlparse(it["url"]).netloc.lower()
        if host.startswith("www."):
            host = host[4:]
        if bucket.get(host, 0) < per_domain:
            out.append(it)
            bucket[host] = bucket.get(host, 0) + 1
    return out

def _fallback_fetch_from_domains(question: str, allowed_domains: List[str], max_snippets: int = 4) -> List[Dict[str, Any]]:
    if not allowed_domains:
        return []
    seeds = []
    for d in allowed_domains[: max_snippets * 2]:
        seeds.append(("https://" + d, d))
        seeds.append(("https://" + d + "/about", d))
    snippets: List[Dict[str, Any]] = []
    for url, d in seeds:
        if _looks_heavy(url):
            continue
        html = _http_get(url)
        if not html:
            continue
        txt = _extract_readable_text(html)
        if not txt:
            continue
        key = (question or "").split()[0].lower() if question else ""
        if key and key not in txt.lower() and len(snippets) >= max_snippets:
            continue
        snippets.append({
            "title": url,
            "url": url,
            "domain": d,
            "content": txt
        })
        if len(_dedupe_by_domain(snippets, per_domain=1)) >= max_snippets:
            break
    snippets = _dedupe_by_domain(snippets, per_domain=1)[:max_snippets]
    _log(f"Fallback produced {len(snippets)} snippet(s).")
    return snippets

# People-authored search (bounded & light)

def _authorship_score(html: str, author_name: str) -> float:
    if not html or not author_name:
        return 0.0
    name = author_name.lower()
    score = 0.0
    try:
        soup = BeautifulSoup(html, "html.parser")
        title = (soup.title.string or "").lower() if soup.title and soup.title.string else ""
        if name in title:
            score += 0.6
        meta_by_author = any(
            (m.get("content","") or "").lower().find(name) >= 0
            for m in soup.find_all("meta", attrs={"name": re.compile(r"(author|byline)", re.I)})
        )
        if meta_by_author:
            score += 0.3
        text = soup.get_text(" ", strip=True).lower()
        if re.search(rf"\bby\s+{re.escape(name)}\b", text):
            score += 0.3
    except Exception:
        pass
    return min(score, 1.0)

def _sample_names_for_question(author_names: List[str], k: int) -> List[str]:
    # deterministic pick based on question hash for stability across runs
    if not author_names:
        return []
    h = int(hashlib.sha1(("|".join(author_names)).encode("utf-8")).hexdigest(), 16)
    start = h % max(1, len(author_names))
    return (author_names[start:] + author_names[:start])[:k]

def _serper_people_search(question: str, allowed_domains: List[str], author_names: List[str], num_per_author: int = 3) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    if not SERPER_API_KEY or not allowed_domains or not author_names:
        return out
    # only consider up to 5 author names to bound queries
    names_considered = _sample_names_for_question(author_names, k=5)
    site_clause = " OR ".join([f"site:{d}" for d in allowed_domains[:12]])
    for name in names_considered:
        q = f'{question} "{name}" {site_clause}'.strip()
        hits = _serper_search_raw(q, num=min(num_per_author, 3))
        for h in hits:
            if _looks_heavy(h["url"]):
                continue
            if _is_allowed_domain(h["url"], allowed_domains):
                h2 = dict(h)
                h2["_author"] = name
                out.append(h2)
    return out

def _people_snippets_from_hits(hits: List[Dict[str, Any]], per_domain: int, max_keep: int) -> List[Dict[str, Any]]:
    enriched = []
    for it in hits:
        url = it["url"]
        html = _http_get(url)
        if not html:
            continue
        score = _authorship_score(html, it.get("_author",""))
        if score <= 0:
            continue
        txt = _extract_readable_text(html)
        if not txt:
            continue
        dom = urlparse(url).netloc.lower()
        if dom.startswith("www."): dom = dom[4:]
        enriched.append({
            "title": it.get("title") or url,
            "url": url,
            "domain": dom,
            "content": txt,
            "_author": it.get("_author",""),
            "_score": score,
        })
    enriched.sort(key=lambda x: x.get("_score",0), reverse=True)
    deduped = _dedupe_by_domain(enriched, per_domain=per_domain)
    return deduped[:max_keep]

def _get_trusted_web_snippets_mixed(
    question: str,
    max_w: int,
    max_p: int,
    per_domain: int
) -> dict:
    """
    Returns {"domain":[...W#...], "people":[...P#...]} with a HARD CAP of WEB_TOTAL_MAX.
    Skips heavy URLs (.pdf/.xml) and caches non-HTML as empty string to avoid refetch.
    """
    domains = [d.strip().lower() for d in (TRUSTED_SOURCES.get("domains") or []) if d.strip()]
    names   = [n.strip() for n in (TRUSTED_SOURCES.get("names") or []) if n.strip()]
    _log(f"Trusted domains={len(domains)}; names={len(names)}; SERPER? {bool(SERPER_API_KEY)}; limits W={max_w} P={max_p} TOTAL={WEB_TOTAL_MAX}")

    if not domains:
        return {"domain": [], "people": []}

    cache_key = (question.strip(), int(max_w), int(max_p), int(per_domain), int(WEB_TOTAL_MAX), tuple(domains), tuple(names))
    if cache_key in _WEB_SNIP_CACHE:
        _log("Web snippet cache hit (mixed).")
        return _WEB_SNIP_CACHE[cache_key]

    out = {"domain": [], "people": []}

    # 1) Domains (W#)
    w_target = min(max_w, WEB_TOTAL_MAX)
    if w_target > 0:
        hits = _serper_search_domains(question, domains, num=w_target * 3 or 6)
        if hits:
            hits = _dedupe_by_domain(hits, per_domain=per_domain)[:w_target]
            w_snips = []
            for it in hits:
                url = it["url"]
                if _looks_heavy(url):
                    continue
                html = _http_get(url)
                if not html:
                    continue
                txt  = _extract_readable_text(html)
                if not txt:
                    continue
                dom = urlparse(url).netloc.lower()
                if dom.startswith("www."): dom = dom[4:]
                w_snips.append({
                    "title": it.get("title") or url,
                    "url": url,
                    "domain": dom,
                    "content": txt
                })
                if len(w_snips) >= w_target:
                    break
            out["domain"] = w_snips
        if not out["domain"]:
            _log("No Serper W# results/snippets; using fallback direct fetch.")
            out["domain"] = _fallback_fetch_from_domains(question, domains, max_snippets=w_target)

    # 2) People-authored (P#) — respect global TOTAL cap
    remaining = max(0, WEB_TOTAL_MAX - len(out["domain"]))
    p_target = min(max_p, remaining)
    if p_target > 0 and names:
        raw_hits = _serper_people_search(question, domains, names, num_per_author=2)
        p_snips = _people_snippets_from_hits(raw_hits, per_domain=per_domain, max_keep=p_target)
        out["people"] = p_snips

    _WEB_SNIP_CACHE[cache_key] = out
    _log(f"Returning W={len(out['domain'])} and P={len(out['people'])} snippet(s). (TOTAL ≤ {WEB_TOTAL_MAX})")
    return out

# Main answer generation

def generate_answer(question: str) -> Dict[str, Any]:
    kb_hits    = retrieve_kb(question, top_k=KB_TOP_K)   # None => unlimited K#
    style_hits = retrieve_style(question, top_k=STYLE_TOP_K)

    mixed = _get_trusted_web_snippets_mixed(
        question=question,
        max_w=WEB_DOMAIN_MAX,
        max_p=WEB_PEOPLE_MAX,
        per_domain=WEB_PER_DOMAIN,
    )
    web_domain_snippets = mixed.get("domain", [])
    web_people_snippets = mixed.get("people", [])

    messages = compose_messages(
        question=question,
        kb_hits=kb_hits,
        style_hits=style_hits,
        web_domain_snippets=web_domain_snippets,
        web_people_snippets=web_people_snippets,
    )

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.2,
        max_tokens=800,
    )
    answer = (resp.choices[0].message.content or "").strip()

    return {
        "answer": answer,
        "kb_hits": kb_hits,
        "web_domain_snippets": web_domain_snippets,
        "web_people_snippets": web_people_snippets,
        "g_tags": [],
    }

In [ ]:
ask("How can HR ensure responsible and ethical use of AI in the workplace?")

[WEB] Trusted domains=13; names=274; SERPER? True; limits W=5 P=3 TOTAL=5
[WEB] SKIP non-HTML https://knowledge.insead.edu/leadership-organisations/how-ai-transforming-hr → 403 text/html; charset=utf-8
[WEB] Returning W=4 and P=0 snippet(s). (TOTAL ≤ 5)

Assistant:

To ensure responsible and ethical use of AI in the workplace, HR can take several key actions:

1. **Address Ethical Concerns**: HR leaders recognize that ethical and security concerns are significant barriers to the adoption of AI technologies like Generative AI (GenAI) in the workplace. It is crucial to establish guidelines that prioritize ethical considerations in AI deployment, ensuring that AI systems are fair, transparent, and accountable [K1], [W3].

2. **Promote Training and Education**: Providing training for employees on how to use AI responsibly is essential. This includes educating them about the limitations and potential biases of AI systems, which can help mitigate fears and build confidence in using these tec